In [0]:
%pip install simple-salesforce pyyaml
dbutils.library.restartPython()

In [0]:
%run ./salesforce

In [0]:
import datetime
import traceback
import pandas
import warnings
import re
from datetime import timezone
from pyspark.sql import SparkSession

warnings.filterwarnings("ignore", category=FutureWarning)
spark = SparkSession.builder.getOrCreate()

In [0]:
BASE_PATH = "/Workspace/Users/e713362@edp.pt/sf_databricks_pbi/data/current"
STATE_PATH = "/Workspace/Users/e713362@edp.pt/sf_databricks_pbi/data/logs/state.json"
SCHEMA = "sf_databricks_pbi"

In [0]:
def sanitizar_dataframe(df):
    df = df.copy()
    df.columns = [
        col.replace(" ", "_").replace("&", "_").replace(",", "_")
        for col in df.columns
    ]

    for col in df.columns:
        if df[col].apply(type).nunique() > 2:
            df[col] = df[col].astype(str).replace("None", None).replace("nan", None)

        if df[col].apply(lambda x: isinstance(x, (dict, list))).any():
            df[col] = df[col].apply(
                lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x
            )
    return df


In [0]:
def guardar_tabela(df_pandas, nome_tabela):
    if df_pandas is None or df_pandas.empty:
        print(f"⚠ {nome_tabela}: sem dados.")
        return

    df_pandas = sanitizar_dataframe(df_pandas)
    df_spark = spark.createDataFrame(df_pandas)

    df_spark.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{SCHEMA}.{nome_tabela}")

    print(f"✔ '{nome_tabela}' guardada ({df_pandas.shape[0]} registos)")

In [0]:
def ler_state():
    """Reads the last sync timestamp."""
    try:
        with open(STATE_PATH, "r") as f:
            state = json.load(f)
        return state.get("last_sync")
    except FileNotFoundError:
        return None

def guardar_state():
    """Saves current UTC timestamp as last sync."""
    state = {"last_sync": datetime.datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000+0000")}
    with open(STATE_PATH, "w") as f:
        json.dump(state, f)
    print(f"✔ State guardado: {state['last_sync']}")

In [0]:
def upsert_tabela(df_pandas, nome_tabela, chave):
    """
    Merges new/updated records into existing Delta table.
    Falls back to overwrite if table doesn't exist yet.
    """
    if df_pandas is None or df_pandas.empty:
        print(f"⚠ {nome_tabela}: sem dados.")
        return

    df_pandas = sanitizar_dataframe(df_pandas)
    df_spark = spark.createDataFrame(df_pandas)
    tabela_completa = f"{SCHEMA}.{nome_tabela}"

    if spark.catalog.tableExists(tabela_completa):
        delta_table = DeltaTable.forName(spark, tabela_completa)

        if isinstance(chave, list):
            condicao = " AND ".join([f"target.{c} = source.{c}" for c in chave])
        else:
            condicao = f"target.{chave} = source.{chave}"

        delta_table.alias("target") \
            .merge(df_spark.alias("source"), condicao) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()

        print(f"✔ '{tabela_completa}' atualizada (merge) com {df_pandas.shape[0]} registos.")
    else:
        df_spark.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(tabela_completa)

        print(f"✔ '{tabela_completa}' criada com {df_pandas.shape[0]} registos.")

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS sf_databricks_pbi")

In [0]:
def main():
    print("Conectando ao Salesforce...")

    try:
        sf = ligacao_salesforce()
        if sf is None:
            print("Erro na ligação.")
            return

        print("Ligação estabelecida com sucesso.")

        lista_ids = ids_ativos(sf)
        if not lista_ids:
            print("Sem ativos.")
            return

        print(f"Número de ativos: {len(lista_ids)}")

        # Incremental
        last_sync = ler_state()
        if last_sync:
            print(f"Última sincronização: {last_sync}")
        else:
            print("Primeira execução — carga completa.")

        periodo = [
            datetime.date.today() - datetime.timedelta(days=3000),
            datetime.date.today()
        ]

        # Ativos
        ativos = obter_ativos(sf_=sf, lista=lista_ids)
        guardar_tabela(ativos, "ativos")

        # Eventos
        eventos = obter_eventos(
            sf_=sf,
            lista=lista_ids,
            periodo=periodo,
            meio=[],
            estado=[],
            desde=last_sync
        )

        # Pedidos de Operação
        pedidos_operacao = obter_po(
            sf_=sf,
            lista=lista_ids,
            meio=[],
            estado=[],
            desde=last_sync
        )

        # Task Owner
        eventos = construir_task_owner(eventos, pedidos_operacao)

        upsert_tabela(eventos, "eventos", chave="case_id")
        upsert_tabela(pedidos_operacao, "pedidos_operacao", chave="case_id")

        # Empresa
        empresa = obter_empresa(sf_=sf, lista=lista_ids)
        guardar_tabela(empresa, "empresa")

        # CT + Equipamentos + Tomadas
        ct, eq, tomadas = obter_ct_eq_tomadas(sf_=sf, lista=lista_ids)
        guardar_tabela(ct, "ct_me")
        guardar_tabela(eq, "equipamentos_me")
        guardar_tabela(tomadas, "tomadas_me")

        # Sintomas dos Eventos Pai
        if eventos is not None and not eventos.empty and "case_id" in eventos.columns:
            lista_cases = eventos["case_id"].dropna().astype(str).unique().tolist()
        elif eventos is not None and not eventos.empty and "Id" in eventos.columns:
            lista_cases = eventos["Id"].dropna().astype(str).unique().tolist()
        else:
            lista_cases = []

        if lista_cases:
            sintomas_eventos_pai = obter_sintoma_eventos_pai(sf_=sf, lista_solicitacoes=lista_cases)
        else:
            sintomas_eventos_pai = pandas.DataFrame()

        upsert_tabela(sintomas_eventos_pai, "sintomas_eventos_pai", chave="case_id")
        guardar_state()
        print("Pipeline concluído com sucesso.")

    except Exception as e:
        print("Erro durante execução:")
        print(e)
        traceback.print_exc()

In [0]:
main()

In [0]:
tabelas = [
    "ativos",
    "ct_me",
    "empresa",
    "equipamentos_me",
    "eventos",
    "pedidos_operacao",
    "sintomas_eventos_pai",
    "tomadas_me"
]

for tabela in tabelas:
    print(f"\n{'='*60}")
    print(f"  {tabela.upper()}")
    print(f"{'='*60}")
    
    # Row count
    count = spark.sql(f"SELECT COUNT(*) as n FROM sf_databricks_pbi.{tabela}").collect()[0]["n"]
    print(f"  Registos: {count}")
    
    # Schema
    print(f"\n  Colunas:")
    colunas = spark.sql(f"DESCRIBE sf_databricks_pbi.{tabela}").toPandas()
    for _, row in colunas.iterrows():
        print(f"    {row['col_name']:40s} {row['data_type']}")
    
    # Preview
    print(f"\n  Preview (5 registos):")
    spark.sql(f"SELECT * FROM sf_databricks_pbi.{tabela} LIMIT 15").show(truncate=30)
